In [0]:
import sys
sys.path.append("/Workspace/Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src")

from data_quality.data_quality_framework import DataQualityRule, run_data_quality_rule, null_check, duplicate_check

In [0]:
from data_quality.quarantine import prepare_quarantine_records

In [0]:
from pyspark.sql import functions as F
import uuid

In [0]:
test_quarantine_df = spark.createDataFrame(
    [
        ("M1001", "A1001"),
        ("M1002", None),
        ("M1003", "A1003"),
    ],
    ["maintenance_id", "aircraft_id"]
)

display(test_quarantine_df)

In [0]:
invalid_records_df = test_quarantine_df.filter(
    F.col("aircraft_id").isNull()
)

display(invalid_records_df)

In [0]:
quarantine_test_run_id = str(uuid.uuid4())

print(quarantine_test_run_id)

In [0]:
QUARANTINE_TABLE = "workspace.aeropulse_dev.dq_quarantine"

print(QUARANTINE_TABLE)

In [0]:
quarantine_df = prepare_quarantine_records(
    invalid_df=invalid_records_df,
    rule_name="aircraft_id_not_null",
    failure_reason="Aircraft ID must not be NULL",
    pipeline_run_id=quarantine_test_run_id,
    source_system="maintenance_app",
    source_entity="maintenance",
    quarantine_table=QUARANTINE_TABLE,
)

display(quarantine_df)

In [0]:
quarantine_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(QUARANTINE_TABLE)

In [0]:
verified_quarantine_df = spark.table(
    QUARANTINE_TABLE
)

display(verified_quarantine_df)

In [0]:
display(
    verified_quarantine_df.filter(
        F.col("rule_name") == "aircraft_id_not_null"
    )
)

In [0]:
quarantine_count = (
    verified_quarantine_df
    .filter(
        F.col("rule_name") == "aircraft_id_not_null"
    )
    .count()
)

print(f"Quarantined records: {quarantine_count}")

In [0]:
quarantine_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(QUARANTINE_TABLE)

In [0]:
display(
    spark.table(QUARANTINE_TABLE)
    .groupBy("rule_name")
    .count()
)

In [0]:
test_quarantine_df = spark.createDataFrame(
    [
        ("M1001", "A1001"),
        ("M1002", None),
        ("M1003", "A1003"),
    ],
    ["maintenance_id", "aircraft_id"]
)

invalid_records_df = test_quarantine_df.filter(
    F.col("aircraft_id").isNull()
)

display(invalid_records_df)

In [0]:
quarantine_test_run_id = str(uuid.uuid4())

print(quarantine_test_run_id)

In [0]:
quarantine_df_1 = prepare_quarantine_records(
    invalid_df=invalid_records_df,
    rule_name="aircraft_id_not_null",
    failure_reason="Aircraft ID must not be NULL",
    pipeline_run_id=quarantine_test_run_id,
    source_system="maintenance_app",
    source_entity="maintenance",
    quarantine_table=QUARANTINE_TABLE,
)

display(
    quarantine_df_1.select(
        "quarantine_id",
        "quarantine_event_id",
        "pipeline_run_id",
        "rule_name",
        "record_json",
    )
)

In [0]:
quarantine_df_2 = prepare_quarantine_records(
    invalid_df=invalid_records_df,
    rule_name="aircraft_id_not_null",
    failure_reason="Aircraft ID must not be NULL",
    pipeline_run_id=quarantine_test_run_id,
    source_system="maintenance_app",
    source_entity="maintenance",
    quarantine_table=QUARANTINE_TABLE,
)

display(
    quarantine_df_2.select(
        "quarantine_id",
        "quarantine_event_id",
        "pipeline_run_id",
        "rule_name",
        "record_json",
    )
)

In [0]:
event_id_1 = (
    quarantine_df_1
    .select("quarantine_event_id")
    .first()["quarantine_event_id"]
)

event_id_2 = (
    quarantine_df_2
    .select("quarantine_event_id")
    .first()["quarantine_event_id"]
)

print("Event ID 1:", event_id_1)
print("Event ID 2:", event_id_2)
print("Same event ID:", event_id_1 == event_id_2)